In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

from ugdatalab import (
    GaiaQuality,
    Local,
    StrictGBPRP,
    Cut1,
    Cut2,
    MixtureContaminationModel,
    prepare_relation_data,
    estimate_initial_theta0,
    fit_relation_mh,
    fit_relation_nuts,
    plot_mollweide,
    plot_period_abs_mag,
    plot_period_luminosity_diff,
    plot_hr,
    plot_inlier_prob_map,
    plot_posterior,
    plot_trace,
    plot_corner,
)

def add_rrlyrae_period_columns(data, *, rrd_period="pf"):
    pf = np.asarray(np.ma.filled(data["pf"], np.nan), dtype=float)
    p1_o = np.asarray(np.ma.filled(data["p1_o"], np.nan), dtype=float)
    classifications = np.asarray(data["best_classification"], dtype=str)

    rrab = classifications == "RRab"
    rrc = classifications == "RRc"
    rrd = classifications == "RRd"

    period = np.full(len(data), np.nan, dtype=float)
    period[rrab] = pf[rrab]
    period[rrc] = p1_o[rrc]
    if rrd_period == "pf":
        period[rrd] = pf[rrd]
    elif rrd_period == "p1_o":
        period[rrd] = p1_o[rrd]
    else:
        raise ValueError("rrd_period must be 'pf' or 'p1_o'.")

    other = ~(rrab | rrc | rrd)
    period[other] = np.where(np.isfinite(pf[other]), pf[other], p1_o[other])

    out = data.copy()
    out["period"] = period
    out["is_rrab"] = rrab
    out["is_rrc"] = rrc
    out["is_rrd"] = rrd
    return out

# 12 & 13

### First, we’ll need to use Gaia distances to estimate the absolute magnitude of RR Lyraes. This will only work if there isn’t a lot of dust between us and the RR Lyraes.
## Explain why this is.

## Plot the distribution of targets you obtained in Galactic coordinates. Verify that your ADQL query has removed stars in the Galactic disk.


The Gaia G-band magnitude is defined from the measured flux $F$ and a photometric zero-point $Z_\mathrm{G}$:

$$G = -2.5 \log_{10}(F) + Z_\mathrm{G}$$

Propagating the flux measurement uncertainty $\sigma_F$ via $\sigma_G^{\rm meas} = \left|\frac{\partial G}{\partial F}\right| \sigma_F$:

$$\sigma_G^{\rm meas} = \frac{2.5}{\ln 10} \frac{\sigma_F}{F}$$

The zero-point also has its own uncertainty $\sigma_{Z_\mathrm{G}}$, which is independent of the flux measurement, so the two contributions add in quadrature:

$$\sigma_G = \sqrt{\left(\sigma_G^{\rm meas}\right)^2 + \sigma_{Z_\mathrm{G}}^2}$$


For a star with parallax $\varpi$ (in mas), the distance is $d = 1/\varpi$ kpc $= 10^3/\varpi$ pc. Substituting into the distance modulus $\mu = 5\log_{10}(d/10\,\text{pc})$:

$$\mu = 5\log_{10}\!\left(\frac{10^3/\varpi}{10}\right) = 5\log_{10}\!\left(\frac{100}{\varpi}\right) = 10 - 5\log_{10}\varpi,$$

and propagating the parallax uncertainty $\sigma_\varpi$ as well,

$$\sigma_\mu = \left|\frac{\partial \mu}{\partial \varpi}\right| \sigma_\varpi = \frac{5}{\ln 10} \frac{\sigma_\varpi}{\varpi}$$

Finally, the absolute magnitude follows directly from $M_G = G - \mu$. Since the photometric and astrometric measurements are independent, they combine in quadrature as well as,

$$\sigma_{M_G} = \sqrt{\sigma_G^2 + \sigma_\mu^2}.$$

In [ ]:
query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
"""


In [ ]:
rrlyrae = GaiaQuality(query)
rrlyrae.data = add_rrlyrae_period_columns(rrlyrae.data)

print(
    f"GaiaQuality sample: RRab={np.count_nonzero(rrlyrae.data['is_rrab']):,}, "
    f"RRc={np.count_nonzero(rrlyrae.data['is_rrc']):,}, "
    f"RRd={np.count_nonzero(rrlyrae.data['is_rrd']):,}"
)

In [ ]:
ax = plot_mollweide(rrlyrae, 
    title=f"Galactic distribution of {len(rrlyrae.data)} RR Lyrae",
    s=10,
)
plt.show()


In [ ]:
rrlyrae = Local(rrlyrae)


In [ ]:
ax = plot_mollweide(rrlyrae, 
    title=f"Galactic distribution of {len(rrlyrae.data)} RR Lyrae",
    s=10,
)
plt.show()


# 14

## Plot period vs. absolute G-band magnitude for all stars returned by your query. Include error bars on your plots.


In [ ]:
ax = plot_period_abs_mag(rrlyrae, 
    title=f"Period–Luminosity plots of {len(rrlyrae.data)} RR Lyrae",
)
plt.show()


# 15

### Apply the quality cuts in Equations C1 and C2 of Lindegren et al. 2018 to the sample.

## Then plot the period-luminosity relation again. Has the scatter decreased?


 Following [Lindegren et al. 2018](https://arxiv.org/abs/1804.09366), a five parameter solution was accepted if the following conditions were met for the source:
1. mean magnitude $G \leq 21.0$
2. `visibility_periods_used` $\geq 6$
3. `astrometric_sigma5d_max` $\leq \left(1.2\,\mathrm{mas}\right) \times \gamma\left(G\right)$

where $\gamma\left(G\right) = \max\left[1,\,10^{0.2\left(G - 18\right)}\right]$. Or alternatively for a fallback solution,
1. `astrometric_matched_observations` $\geq 5$
2. `astrometric_excess_noise` $< 20\,\text{mas}$
3. $\sigma_{\rm pos,\,max} < 100\,\text{mas}$

Additionally, we use further quality cuts:
1. $u < 1.2 \times \max\!\left[1,\, e^{-0.2(G - 19.5)}\right]$
2. $1.0 + 0.015\left(G_\mathrm{BP} - G_\mathrm{RP}\right)^2 < E < 1.3 + 0.06\left(G_\mathrm{BP} - G_\mathrm{RP}\right)^2$

Here, cut 1 removes outliers $G < 6$ caused by uncalibrated CCD saturation and a blob of moderately large values of $u = \sqrt{\chi^2 / \nu}$ for $G > 18$, possibly extending to much larger values for brighter sources; cut 2 provides stricter criterias on the flux excess factor $E = \left(I_\mathrm{BP} + I_\mathrm{RP}\right) / I_\mathrm{G}$ to deal with possible issues with the BP and RP photometry.

In [ ]:
strict_gbprp = StrictGBPRP(rrlyrae)
cut1  = Cut1(rrlyrae)
cut2  = Cut2(rrlyrae)

fig, axes = plt.subplots(1, 3, figsize=(11, 6), dpi=300, sharey=True, sharex=True)

for ax, dataset, label in zip(axes, [strict_gbprp, cut1, cut2], ["Strict G, BP, RP Cuts", "Cut 1", "Cut 2"]):
    plot_period_luminosity_diff(rrlyrae, dataset, ax=ax, title=f"{label} ($N={len(dataset.data)}$)")

fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 6), dpi=300, sharey=True, sharex=True)

for ax, dataset, label in zip(axes, [rrlyrae, strict_gbprp, cut1, cut2], ["Local", "Strict G, BP, RP Cuts", "Cut 1", "Cut 2"]):
    plot_hr(dataset, ax=ax, title=f"{label} ($N={len(dataset.data)}$)")

# Set limits from Local so StrictGBPRP/Cut1/Cut2 don't zoom in on their smaller ranges
bp_rp = np.asarray(rrlyrae.data["bp_rp"])
M_G   = np.asarray(rrlyrae.data["M_G"])
axes[0].set_xlim(np.nanmin(bp_rp), np.nanmax(bp_rp))
axes[0].set_ylim(np.nanmax(M_G),   np.nanmin(M_G))   # inverted

fig.tight_layout()
plt.show()


In [ ]:
rrlyrae = Cut2(Cut1(StrictGBPRP(rrlyrae)))


In [ ]:
fig = plt.figure(figsize=(12, 5), dpi=300)
gs  = gridspec.GridSpec(1, 2, figure=fig)

ax0 = fig.add_subplot(gs[0], projection="mollweide")
plot_mollweide(rrlyrae, 
    ax=ax0,
    title=f"Galactic distribution of {len(rrlyrae.data)} RR Lyrae",
    s=10,
)

ax1 = fig.add_subplot(gs[1])
plot_period_abs_mag(rrlyrae, 
    ax=ax1,
    title=f"Period–Luminosity plots of {len(rrlyrae.data)} RR Lyrae",
)

fig.tight_layout()
plt.show()


# 16

### You can crudely remove these outliers by removing objects with absolute G magnitudes greater than some threshold of your choice. You can also try more sophisticated outlier rejection, if you wish.

## Plot the period-absolute magnitude relation, including error bars on absolute magnitude due to distance uncertainties, from the resulting cleaned sample.


The hard quality cuts (`StrictGBPRP`, `Cut1`, `Cut2`) are astrometric and photometric cuts motivated by the quality selection pipeline detailed in Lindegren et al. 2018 which removes stars with bad measurements (saturated pixels, crowded apertures, poor astrometric fits). However, they cannot identify astrophysical outliers such as misclassified variable stars, unresolved binaries, foreground/background contamination, or stars whose distance errors place them far from the true period–luminosity locus.

We can use a Gaussian mixture model which treats every star probabilistically, being either an inlier drawn from the true period–luminosity relation, or an outlier drawn from a background distribution. Each star receives a posterior probability of being an inlier, and we reject those below a threshold, chosen to be 95% here.

For each subpopulation (RRab and RRc treated separately), let:

$$
x_i = \log_{10}(P_i), \qquad y_i = M_{G,i}, \qquad \sigma_i = \sigma_{M,i}
$$

The inlier component is a Gaussian along the period–luminosity relation with intrinsic scatter $\sigma_\mathrm{scatter}$:

$$
y_i \mid \text{inlier} \sim \mathcal{N}\!\left(a + b\,x_i,\; \sigma_i^2 + \sigma_\mathrm{scatter}^2\right)
$$

The outlier component is a broad Gaussian centred on the sample median with width chosen as $3\sigma$:

$$
y_i \mid \text{outlier} \sim \mathcal{N}\!\left(\mu_\mathrm{bg},\; \sigma_\mathrm{bg}^2\right), \qquad \mu_\mathrm{bg} = \mathrm{median}(y), \qquad \sigma_\mathrm{bg} = 3\,\mathrm{std}(y)
$$

The mixture likelihood for each star, as a weighted sum of two components inlier period-luminosity component and outlier/background component, is

$$
\mathcal{L}_i = f\,\mathcal{N}_\mathrm{in}(y_i) + (1-f)\,\mathcal{N}_\mathrm{out}(y_i), \qquad 0 < f < 1,
$$

where $f$ is the inlier fraction.

Since $f$ must stay between 0 and 1, it is more convenient reparameterize $f$ as $\eta$,

$$
\eta \equiv \operatorname{logit}(f) = \log\!\left(\frac{f}{1-f}\right), \qquad f = \frac{1}{1 + e^{-\eta}}.
$$

This reparameterisation maps the bounded interval $f \in (0,1)$ onto the full real line $\eta \in (-\infty,\infty)$, so the MCMC sampler can propose Gaussian steps in $\eta$ without ever producing an invalid mixture fraction.

Therefore, we sample the unconstrained parameter vector

$$\theta = \bigl(a,\, b,\, \ell_\sigma,\, \eta\bigr), \qquad \ell_\sigma \equiv \log_{10}\sigma_\mathrm{scatter}, \qquad \eta \equiv \operatorname{logit}(f),$$

and select the following priors,

$$a \sim \mathcal{N}(0,10^2), \qquad b \sim \mathcal{N}(0,10^2), \qquad \ell_\sigma \sim \mathcal{N}(0,2^2), \qquad \eta \sim \mathcal{N}(0,3^2).$$

Since for a zero-mean Gaussian prior $x \sim \mathcal{N}(0,s^2)$,

$$\log \pi(x) = -\frac{1}{2}\left(\frac{x}{s}\right)^2 - \log\!\bigl(s\sqrt{2\pi}\bigr).$$

Assuming independence, the joint prior product is $\pi(\theta) = \pi(a)\,\pi(b)\,\pi(\ell_\sigma)\,\pi(\eta)$, so in log-space, the log-prior is the sum of the four Gaussian log-densities. Dropping the additive normalisation constants, which do not affect MCMC acceptance ratios, gives

$$\log \pi(\theta) = -\frac{1}{2}\!\left(\frac{a}{10}\right)^2 - \frac{1}{2}\!\left(\frac{b}{10}\right)^2 - \frac{1}{2}\!\left(\frac{\log_{10}\sigma_\mathrm{scatter}}{2}\right)^2 - \frac{1}{2}\!\left(\frac{\operatorname{logit} f}{3}\right)^2 + \text{const}, $$

and the quantity sampled by MCMC is the unnormalised log-posterior,

$$\log p(\theta \mid \mathbf{y}) = \sum_i \log \mathcal{L}_i + \log \pi(\theta) + \text{const}. $$

Using `emcee` package, we will obtain a posterior inlier probability for each star (i.e. the expectation of the inlier fraction):

$$r_i = \left\langle \frac{f\,\mathcal{N}_\mathrm{in}(y_i \mid \theta)}{f\,\mathcal{N}_\mathrm{in}(y_i \mid \theta) + (1-f)\,\mathcal{N}_\mathrm{out}(y_i)} \right\rangle_{\theta \sim p(\theta \mid \mathbf{y})}.$$

Through the posterior inlier probability, we reject outliers below a threshold

$$
r_i \leq 0.95.
$$


In [ ]:
rrlyrae_clean = MixtureContaminationModel(rrlyrae)

print(f"Kept {len(rrlyrae_clean.data)} / {len(rrlyrae.data)} stars")


In [ ]:
fig = plt.figure(figsize=(12, 5), dpi=300)
gs  = gridspec.GridSpec(1, 2, figure=fig)

ax0 = fig.add_subplot(gs[0])
plot_inlier_prob_map(rrlyrae_clean, 
    ax=ax0,
    title=f"Posterior inlier probability (threshold={rrlyrae_clean.prob_threshold})",
)

ax1 = fig.add_subplot(gs[1])
plot_period_luminosity_diff(rrlyrae, 
    rrlyrae_clean,
    ax=ax1,
    title=f"Mixture model: kept $N={len(rrlyrae_clean.data)}$, "
          f"removed $N={len(rrlyrae.data)-len(rrlyrae_clean.data)}$",
)

fig.tight_layout()
plt.show()

In [ ]:
ax = plot_period_abs_mag(rrlyrae_clean, 
    title=f"Cleaned period–luminosity ({len(rrlyrae_clean.data)} RR Lyrae)",
)
plt.show()


# 18

We now fit the period-luminosity relation separately for RRab and RRc stars.
For each class we use the same linear model,

$$
M_G = a\log_{10}(P/\mathrm{day}) + b,
$$

with intrinsic scatter $\sigma_\mathrm{scatter}$ added in quadrature with the
measurement uncertainty on $M_G$.  The likelihood and priors are the same as in
our RRab-only draft, but every fit below is now class-specific.

In [ ]:
RR_CLASSES = ("RRab", "RRc")
METHOD_META = [
    ("mh", "Metropolis-Hastings", "C0"),
    ("nuts_potential", "NUTS - explicit Potential", "C1"),
    ("nuts_native", "NUTS - native pm.Normal", "C2"),
]
PL_NATIVE_COLORS = {"RRab": "C0", "RRc": "C1"}


def post_burn_samples(sampler):
    return sampler.samples[sampler.n_burn:]


def summarize_relation_samples(samples):
    return {
        "slope_median": np.median(samples[:, 0]),
        "slope_std": np.std(samples[:, 0]),
        "intercept_median": np.median(samples[:, 1]),
        "intercept_std": np.std(samples[:, 1]),
        "sigma_median": 10 ** np.median(samples[:, 2]),
        "sigma_std": np.std(10 ** samples[:, 2]),
    }


def thin_samples(samples, n_keep=8000, seed=0):
    n_keep = min(n_keep, len(samples))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(samples), size=n_keep, replace=False)
    return samples[idx]


pl_data = {
    rr_class: prepare_relation_data(rrlyrae_clean, rr_class, "pl")
    for rr_class in RR_CLASSES
}
pl_theta0 = {
    rr_class: estimate_initial_theta0(
        rrlyrae_clean,
        rr_class,
        "pl",
        data=pl_data[rr_class],
    )
    for rr_class in RR_CLASSES
}

for rr_class in RR_CLASSES:
    data = pl_data[rr_class]
    theta0 = pl_theta0[rr_class]
    print(
        f"{rr_class}: {len(data.x)} stars | initial theta0 = "
        f"[{theta0[0]:.3f}, {theta0[1]:.3f}, {theta0[2]:.3f}]"
    )

In [ ]:
pl_results = {}
seed_map_pl = {"RRab": 42, "RRc": 84}

for rr_class in RR_CLASSES:
    data = pl_data[rr_class]
    theta0 = pl_theta0[rr_class]
    seed = seed_map_pl[rr_class]

    mh_sampler = fit_relation_mh(
        data,
        theta0,
        n_steps=30_000,
        n_burn=5_000,
        seed=seed,
        tune=True,
    )
    nuts_potential = fit_relation_nuts(
        data,
        theta0,
        model_kind="potential",
        n_steps=30_000,
        n_burn=5_000,
        seed=seed,
    )
    nuts_native = fit_relation_nuts(
        data,
        theta0,
        model_kind="native",
        n_steps=30_000,
        n_burn=5_000,
        seed=seed,
    )

    pl_results[rr_class] = {
        "data": data,
        "mh": mh_sampler,
        "nuts_potential": nuts_potential,
        "nuts_native": nuts_native,
    }
    for method_key, method_label, _ in METHOD_META:
        samples = post_burn_samples(pl_results[rr_class][method_key])
        summary = summarize_relation_samples(samples)
        print(
            f"{rr_class} | {method_label}: "
            f"acceptance={pl_results[rr_class][method_key].acceptance_rate:.3f}, "
            f"a={summary['slope_median']:.3f} +/- {summary['slope_std']:.3f}, "
            f"b={summary['intercept_median']:.3f} +/- {summary['intercept_std']:.3f}, "
            f"sigma_scatter={summary['sigma_median']:.3f} mag"
        )

In [ ]:
for rr_class in RR_CLASSES:
    for method_key, method_label, _ in METHOD_META:
        plot_trace(pl_results[rr_class][method_key], 
            title=f"Trace: {rr_class} period-luminosity ({method_label})"
        )
        plt.show()

In [ ]:
for rr_class in RR_CLASSES:
    for method_key, method_label, _ in METHOD_META:
        plot_corner(pl_results[rr_class][method_key], 
            title=f"Posterior: {rr_class} period-luminosity ({method_label})"
        )
        plt.show()

In [ ]:
for rr_class in RR_CLASSES:
    data = pl_results[rr_class]["data"]
    x_range = np.linspace(data.x.min(), data.x.max(), 300)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=300, sharey=True)

    for ax, (method_key, method_label, color) in zip(axes, METHOD_META):
        samples = post_burn_samples(pl_results[rr_class][method_key])
        rng = np.random.default_rng(seed_map_pl[rr_class] + len(method_key))
        idx = rng.choice(len(samples), size=50, replace=False)

        ax.errorbar(
            data.x,
            data.y,
            yerr=data.sigma,
            fmt='.',
            color='k',
            alpha=0.35,
            ms=3,
            lw=0.5,
            zorder=1,
            label=f"{rr_class} data",
        )
        for slope, intercept, _ in samples[idx]:
            ax.plot(x_range, slope * x_range + intercept, lw=0.5, alpha=0.25, color=color)
        ax.plot([], [], color=color, lw=1.5, label='50 posterior draws')
        ax.set_title(f"{rr_class}: {method_label}")
        ax.set_xlabel(data.x_label)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    axes[0].set_ylabel(data.y_label)
    axes[0].invert_yaxis()
    fig.suptitle(f"Posterior predictive check: {rr_class} period-luminosity relation")
    plt.tight_layout()
    plt.show()

In [ ]:
import corner
from matplotlib.lines import Line2D

for rr_class in RR_CLASSES:
    overlay = None
    for offset, (method_key, method_label, color) in enumerate(METHOD_META):
        samples = thin_samples(
            post_burn_samples(pl_results[rr_class][method_key]),
            seed=seed_map_pl[rr_class] + offset,
        )
        overlay = corner.corner(
            samples,
            fig=overlay,
            labels=[r"$a$", r"$b$", r"$\log_{10}\sigma_\mathrm{scatter}$"],
            show_titles=False,
            quantiles=[0.16, 0.5, 0.84],
            color=color,
        )

    handles = [
        Line2D([0], [0], color=color, lw=2, label=method_label)
        for _, method_label, color in METHOD_META
    ]
    overlay.legend(handles=handles, loc="upper right", fontsize=9)
    overlay.suptitle(
        f"Posterior comparison: {rr_class} period-luminosity relation\n"
        "methods (i) MH, (ii) NUTS Potential, (iii) NUTS native",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

# 22 - Comparison to Literature and RRab/RRc Differences

We keep the literature comparison for RRab stars, where the reference
calibrations are most directly comparable, and add a direct RRab-vs-RRc
comparison using the class-specific native-NUTS fits.

In [ ]:
rrab_native = post_burn_samples(pl_results["RRab"]["nuts_native"])
rrc_native = post_burn_samples(pl_results["RRc"]["nuts_native"])

print("Posterior summary - native NUTS period-luminosity fits")
print(f"{'Class':<6} {'a_med':>10} {'a_std':>10} {'b_med':>10} {'b_std':>10} {'sigma_med':>12}")
for rr_class, samples in [("RRab", rrab_native), ("RRc", rrc_native)]:
    summary = summarize_relation_samples(samples)
    print(
        f"{rr_class:<6} "
        f"{summary['slope_median']:>10.4f} {summary['slope_std']:>10.4f} "
        f"{summary['intercept_median']:>10.4f} {summary['intercept_std']:>10.4f} "
        f"{summary['sigma_median']:>12.4f}"
    )

literature = {
    "Sesar+2017 (V-band)": {"a": -1.62, "b": 2.32, "color": "C4", "ls": "--"},
    "Neeley+2019 (G-band)": {"a": -2.04, "b": 0.49, "color": "C3", "ls": "-."},
}

x_rrab = np.linspace(pl_results["RRab"]["data"].x.min(), pl_results["RRab"]["data"].x.max(), 300)
rng_rrab = np.random.default_rng(3)
idx_rrab = rng_rrab.choice(len(rrab_native), size=50, replace=False)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
rrab_data = pl_results["RRab"]["data"]
ax.errorbar(
    rrab_data.x,
    rrab_data.y,
    yerr=rrab_data.sigma,
    fmt='.',
    color='k',
    alpha=0.35,
    ms=3,
    lw=0.5,
    zorder=1,
    label='RRab data',
)
for slope, intercept, _ in rrab_native[idx_rrab]:
    ax.plot(x_rrab, slope * x_rrab + intercept, lw=0.5, alpha=0.25, color='C2', zorder=2)
ax.plot([], [], color='C2', lw=1.5, label='50 RRab native-NUTS draws')
for label, lit in literature.items():
    ax.plot(x_rrab, lit['a'] * x_rrab + lit['b'], color=lit['color'], ls=lit['ls'], lw=1.8, label=label)
ax.set_xlabel(rrab_data.x_label)
ax.set_ylabel(rrab_data.y_label)
ax.set_title('RRab period-luminosity relation: this work vs. literature')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
for rr_class in RR_CLASSES:
    data = pl_results[rr_class]["data"]
    samples = post_burn_samples(pl_results[rr_class]["nuts_native"])
    rng = np.random.default_rng(seed_map_pl[rr_class] + 200)
    idx = rng.choice(len(samples), size=50, replace=False)
    x_range = np.linspace(data.x.min(), data.x.max(), 300)
    color = PL_NATIVE_COLORS[rr_class]

    ax.errorbar(
        data.x,
        data.y,
        yerr=data.sigma,
        fmt='.',
        color=color,
        alpha=0.30,
        ms=3,
        lw=0.5,
        zorder=1,
        label=f'{rr_class} data',
    )
    for slope, intercept, _ in samples[idx]:
        ax.plot(x_range, slope * x_range + intercept, lw=0.5, alpha=0.20, color=color, zorder=2)
    summary = summarize_relation_samples(samples)
    ax.plot(
        x_range,
        summary['slope_median'] * x_range + summary['intercept_median'],
        color=color,
        lw=2.0,
        label=f"{rr_class} median fit",
        zorder=3,
    )

ax.set_xlabel(rrab_data.x_label)
ax.set_ylabel(rrab_data.y_label)
ax.set_title('Native-NUTS period-luminosity fits: RRab vs. RRc')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# 23 - Period-Color Relation

We also fit the intrinsic Gaia color relation separately for RRab and RRc stars:

$$
(G_\mathrm{BP} - G_\mathrm{RP}) = a_c\log_{10}(P/\mathrm{day}) + b_c,
$$

with intrinsic color scatter added in quadrature with the photometric color
uncertainty.  Because color is distance-independent, these fits are the ones we
will carry into the dust-map notebook.

In [ ]:
pc_data = {
    rr_class: prepare_relation_data(rrlyrae_clean, rr_class, "pc")
    for rr_class in RR_CLASSES
}
pc_theta0 = {
    rr_class: estimate_initial_theta0(
        rrlyrae_clean,
        rr_class,
        "pc",
        data=pc_data[rr_class],
    )
    for rr_class in RR_CLASSES
}

pc_results = {}
seed_map_pc = {"RRab": 142, "RRc": 184}

for rr_class in RR_CLASSES:
    data = pc_data[rr_class]
    theta0 = pc_theta0[rr_class]
    sampler = fit_relation_nuts(
        data,
        theta0,
        model_kind="native",
        n_steps=10_000,
        n_burn=2_000,
        seed=seed_map_pc[rr_class],
    )
    samples = post_burn_samples(sampler)
    pc_results[rr_class] = {
        "data": data,
        "nuts_native": sampler,
        "samples": samples,
        "summary": summarize_relation_samples(samples),
    }
    summary = pc_results[rr_class]["summary"]
    print(
        f"{rr_class} period-color: N={len(data.x)}, acceptance={sampler.acceptance_rate:.3f}, "
        f"a_c={summary['slope_median']:.3f} +/- {summary['slope_std']:.3f}, "
        f"b_c={summary['intercept_median']:.3f} +/- {summary['intercept_std']:.3f}, "
        f"sigma_c={summary['sigma_median']:.4f} mag"
    )

In [ ]:
for rr_class in RR_CLASSES:
    plot_trace(pc_results[rr_class]["nuts_native"], 
        title=f"Trace: {rr_class} period-color relation (native NUTS)"
    )
    plt.show()
    plot_corner(pc_results[rr_class]["nuts_native"], 
        title=f"Posterior: {rr_class} period-color relation (native NUTS)"
    )
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=300, sharey=True)

for ax, rr_class in zip(axes, RR_CLASSES):
    data = pc_results[rr_class]["data"]
    samples = pc_results[rr_class]["samples"]
    x_range = np.linspace(data.x.min(), data.x.max(), 300)
    rng = np.random.default_rng(seed_map_pc[rr_class] + 20)
    idx = rng.choice(len(samples), size=50, replace=False)
    color = PL_NATIVE_COLORS[rr_class]

    ax.errorbar(
        data.x,
        data.y,
        yerr=data.sigma,
        fmt='.',
        color='k',
        alpha=0.35,
        ms=3,
        lw=0.5,
        zorder=1,
        label=f'{rr_class} data',
    )
    for slope, intercept, _ in samples[idx]:
        ax.plot(x_range, slope * x_range + intercept, lw=0.5, alpha=0.25, color=color, zorder=2)
    ax.plot([], [], color=color, lw=1.5, label='50 posterior draws')
    ax.set_title(f"{rr_class} period-color relation")
    ax.set_xlabel(data.x_label)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel(pc_results['RRab']['data'].y_label)
fig.suptitle('Posterior predictive check: class-specific period-color relations')
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

legacy_path = Path('flat_pc.npy')
legacy_path.unlink(missing_ok=True)

pc_output_paths = {
    'RRab': Path('../archive/flat_pc_rrab.npy'),
    'RRc': Path('../archive/flat_pc_rrc.npy'),
}

for rr_class, path in pc_output_paths.items():
    np.save(path, pc_results[rr_class]['samples'])
    print(f"Saved {path.name} shape={pc_results[rr_class]['samples'].shape}")